# 02 — Official labels & splits, full exploration (issue #3)

Uses the **full official untrimmed EHRSHOT dataset** (`EHRSHOT_ASSETS` + `meds_reader_omop_ehrshot`), not the professor's trimmed OMOP copy — this notebook needs the real benchmark labels for all 15 released tasks, the official `person_id_map.csv` splits, and the real OMOP `concept.csv` (5.9M rows, bundled inside `EHRSHOT_ASSETS/femr/logs/omop_dir/`) to resolve diagnosis codes to actual disease names.

**Structure:**
1. Every released benchmark task, briefly — not just our two locked tasks.
2. Validate the two locked tasks (`guo_los`, `new_hypertension`) against the counts in `PROJECT_GUIDE.md`.
3. Zoom in on each locked task: class balance, temporal coverage, demographics, top diagnoses (real names, not codes), a diagnosis co-occurrence network preview, and where the two classes sit in CLMBR embedding space.

**One important caveat, stated once here instead of on every cell:** all of Part 3's diagnosis/demographic exploration uses each patient's **full history**, not history censored to before their `prediction_time`. That's fine for descriptive exploration — it is *not* leakage-safe, and none of these numbers should be read as a model result. The leakage-safe condition is built separately in issues #6/#7/#8.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import itertools

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

from ehrgraph.data.cohort import demographics_for_subjects, patient_code_counts
from ehrgraph.data.loader import open_database
from ehrgraph.data.vocabulary import load_concept_table, resolve_codes
from ehrgraph.eval.baseline import load_clmbr_features
from ehrgraph.eval.labels import summarize_all_tasks

plt.rcParams["figure.facecolor"] = "white"

## Part 1 — every released benchmark task, briefly

In [ ]:
task_summary = summarize_all_tasks(assets)
task_summary_sorted = task_summary.sort_values("n_patients", ascending=False)
task_summary_sorted[["task", "n_rows", "n_patients", "label_type", "positive_rate"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(task_summary_sorted["task"], task_summary_sorted["n_patients"], color="#4C72B0")
axes[0].set_xlabel("n patients")
axes[0].set_title("Cohort size per released benchmark task")
axes[0].invert_yaxis()

boolean_tasks = task_summary_sorted[task_summary_sorted["label_type"] == "boolean"]
axes[1].barh(boolean_tasks["task"], boolean_tasks["positive_rate"], color="#DD8452")
axes[1].set_xlabel("positive rate")
axes[1].set_title("Class balance -- boolean tasks only")
axes[1].invert_yaxis()
for task in ("guo_los", "new_hypertension"):
    axes[0].get_yticklabels()[list(task_summary_sorted["task"]).index(task)].set_color("crimson")

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(task_summary_sorted["task"], task_summary_sorted["n_rows"], color="#55A868")
ax.set_yscale("log")
ax.set_ylabel("n label rows (log scale)")
ax.set_title("Label volume per task -- note rows != patients for most tasks")
plt.xticks(rotation=60, ha="right")
fig.tight_layout()
plt.show()

Two very different task families here: the `lab_*` and `chexpert` tasks generate **many label rows per patient** (repeated lab draws / repeated chest X-rays), while the `guo_*` and `new_*` tasks are closer to one-ish rows per patient. That distinction is exactly what Part 2 checks for our two locked tasks specifically, since it changes how load-bearing the (patient_id, prediction_time) identity is downstream.

## Part 2 — validate the two locked tasks against `PROJECT_GUIDE.md`

In [ ]:
splits = pd.read_csv(assets / "splits" / "person_id_map.csv")
LOCKED_TASKS = {
    "guo_los": {"expected_patients": 3855, "expected_positive_patients": 1271},
    "new_hypertension": {"expected_patients": 2328, "expected_positive_patients": 386},
}

validation_rows = []
task_labels = {}
for task, expected in LOCKED_TASKS.items():
    labels = pd.read_csv(
        assets / "benchmark" / task / "labeled_patients.csv", parse_dates=["prediction_time"]
    )
    labels["value_bool"] = labels["value"].astype(str) == "True"
    task_labels[task] = labels

    n_patients = labels["patient_id"].nunique()
    ever_positive = labels.groupby("patient_id")["value_bool"].any()
    n_positive_patients = int(ever_positive.sum())

    validation_rows.append(
        {
            "task": task,
            "n_rows": len(labels),
            "n_patients": n_patients,
            "n_patients_matches_guide": n_patients == expected["expected_patients"],
            "n_positive_patients": n_positive_patients,
            "n_positive_matches_guide": n_positive_patients
            == expected["expected_positive_patients"],
            "rows_equal_patients": len(labels) == n_patients,
        }
    )

validation_df = pd.DataFrame(validation_rows)
validation_df

In [ ]:
assert validation_df["n_patients_matches_guide"].all(), "Patient counts do not match PROJECT_GUIDE"
assert validation_df["n_positive_matches_guide"].all(), (
    "Positive-patient counts do not match PROJECT_GUIDE"
)
assert not validation_df["rows_equal_patients"].any(), (
    "Expected label rows to outnumber patients for both tasks -- a patient can have multiple "
    "prediction_time rows with different values, which is exactly what makes (patient_id, "
    "prediction_time) the real node identity, not patient_id alone."
)
print(
    "Issue #3 core validation: PASSED -- both tasks' patient and positive-patient counts "
    "match PROJECT_GUIDE.md exactly, confirmed against our own copy of the real data."
)

In [ ]:
for task, labels in task_labels.items():
    merged = labels.merge(
        splits.rename(columns={"omop_person_id": "patient_id"}), on="patient_id", how="left"
    )
    print(f"\n{task} -- patients per split:")
    print(merged.groupby("split")["patient_id"].nunique())
    assert merged["split"].isna().sum() == 0, f"{task}: some patients missing a split assignment"

## Part 3 — zoom in on the two locked tasks

Loads the real OMOP `concept.csv` once (5.9M rows, ~7s, ~700MB in memory -- fine for a one-time exploratory notebook run) and reuses it for both tasks' diagnosis-name resolution.

In [ ]:
concept_table = load_concept_table(assets / "femr" / "logs" / "omop_dir" / "concept.csv")
db = open_database(str(meds_reader_path))
db_ids = set(db)
print(f"concept.csv: {len(concept_table):,} rows")
print(f"meds_reader database: {len(db_ids):,} subjects")

In [ ]:
def zoom_in(task: str, build_cooccurrence_network: bool = False):
    """All of Part 3's exploration for one locked task: class balance x split,
    temporal coverage, demographics, top real-named diagnoses by class, CLMBR
    embedding space, and (optionally) a diagnosis co-occurrence network
    preview. Defined once and called per task so the two tasks' sections stay
    in sync rather than being two hand-copied blocks of cells."""
    print(f"\n{'=' * 70}\n{task}\n{'=' * 70}")
    labels = task_labels[task]
    merged = labels.merge(
        splits.rename(columns={"omop_person_id": "patient_id"}), on="patient_id", how="left"
    )

    # --- class balance x split ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ct = merged.groupby(["split", "value_bool"]).size().unstack(fill_value=0)
    ct = ct.reindex(["train", "val", "test"])
    ct.plot(kind="bar", stacked=True, ax=axes[0], color=["#4C72B0", "#C44E52"])
    axes[0].set_title(f"{task}: class balance x split")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=0)

    # --- temporal coverage ---
    labels["prediction_time"].dt.to_period("Y").value_counts().sort_index().plot(
        kind="bar", ax=axes[1], color="#8172B2"
    )
    axes[1].set_title(f"{task}: prediction_time by year")
    axes[1].tick_params(axis="x", rotation=60)

    # --- cohort membership, filtered to meds_reader coverage ---
    patient_ids_all = labels["patient_id"].unique().tolist()
    coverage = sum(p in db_ids for p in patient_ids_all) / len(patient_ids_all)
    ever_positive = set(labels.groupby("patient_id")["value_bool"].any().pipe(lambda s: s[s].index))
    patient_ids = [p for p in patient_ids_all if p in db_ids]
    pos_ids = [p for p in patient_ids if p in ever_positive]
    neg_ids = [p for p in patient_ids if p not in ever_positive]
    print(
        f"meds_reader coverage: {coverage:.4%} ({len(patient_ids)}/{len(patient_ids_all)} "
        f"patients) -- {len(pos_ids)} ever-positive, {len(neg_ids)} ever-negative in coverage"
    )

    # --- demographics: age at each patient's earliest prediction_time ---
    ref_time = labels.groupby("patient_id")["prediction_time"].min().to_dict()
    demo = demographics_for_subjects(db, patient_ids, ref_time)
    demo["ever_positive"] = demo["subject_id"].isin(ever_positive)
    demo.boxplot(column="age_years", by="ever_positive", ax=axes[2])
    axes[2].set_title(f"{task}: age at first prediction_time")
    fig.suptitle("")
    fig.tight_layout()
    plt.show()

    print(demo.groupby("ever_positive")["age_years"].mean().rename("mean age_years"))
    print(demo["sex"].value_counts(dropna=False).rename("sex counts"))

    # --- top diagnoses, real names, split by class ---
    pos_counts = patient_code_counts(db, pos_ids, table="condition")
    neg_counts = patient_code_counts(db, neg_ids, table="condition")
    all_codes = set(pos_counts) | set(neg_counts)
    overall = pd.Series({c: pos_counts.get(c, 0) + neg_counts.get(c, 0) for c in all_codes})

    resolved_all = resolve_codes(pd.Series(list(all_codes)), concept_table).set_index("code")
    match_rate = resolved_all["concept_name"].notna().mean()
    print(
        f"concept.csv name-resolution match rate across all {len(all_codes):,} distinct "
        f"condition codes: {match_rate:.4%}"
    )

    top20 = overall.sort_values(ascending=False).head(20)
    top20_df = pd.DataFrame(
        {
            "name": resolved_all.loc[top20.index, "concept_name"].fillna(top20.index.to_series()),
            "pos_rate": [pos_counts.get(c, 0) / max(len(pos_ids), 1) for c in top20.index],
            "neg_rate": [neg_counts.get(c, 0) / max(len(neg_ids), 1) for c in top20.index],
        }
    ).iloc[::-1]

    fig, ax = plt.subplots(figsize=(9, 7))
    y = np.arange(len(top20_df))
    ax.barh(
        y - 0.2, top20_df["pos_rate"], height=0.4, label="ever-positive cohort", color="#C44E52"
    )
    ax.barh(
        y + 0.2, top20_df["neg_rate"], height=0.4, label="ever-negative cohort", color="#4C72B0"
    )
    ax.set_yticks(y)
    ax.set_yticklabels(top20_df["name"], fontsize=8)
    ax.set_xlabel("fraction of cohort with >=1 event of this diagnosis (full history)")
    ax.set_title(f"{task}: top 20 diagnoses by real name, positive vs negative cohort")
    ax.legend()
    fig.tight_layout()
    plt.show()

    # --- CLMBR embedding space ---
    features = load_clmbr_features(assets / "features" / "clmbr_features.pkl")
    joined = labels.merge(features, on=["patient_id", "prediction_time"], how="inner")
    print(f"CLMBR embedding join rate: {len(joined) / len(labels):.4%}")
    X = np.stack(joined["embedding"].to_numpy())
    pca = PCA(n_components=2, random_state=0)
    Z = pca.fit_transform(X)

    fig, ax = plt.subplots(figsize=(6, 6))
    for value, color, label in [(False, "#4C72B0", "negative"), (True, "#C44E52", "positive")]:
        mask = joined["value_bool"].to_numpy() == value
        ax.scatter(Z[mask, 0], Z[mask, 1], s=6, alpha=0.5, color=color, label=label)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
    ax.set_title(f"{task}: CLMBR embeddings, PCA, colored by label")
    ax.legend()
    fig.tight_layout()
    plt.show()

    if build_cooccurrence_network:
        # Raw shared-patient counts make a near-complete graph here -- the
        # top-30 diagnoses are common enough that most pairs co-occur in a
        # comorbid population by base rate alone. Lift (joint probability /
        # product of marginals) surfaces genuinely elevated associations
        # instead, and keeping only the strongest edges sparsifies the graph
        # into readable clusters -- the same idea issue #5's mutual k-NN
        # sparsify applies to the real patient-similarity graph.
        n_cohort = len(patient_ids)
        top30_codes = overall.sort_values(ascending=False).head(30).index.tolist()
        top30_set = set(top30_codes)
        joint_counts: dict[tuple[str, str], int] = {}
        for pid in patient_ids:
            codes_here = {e.code for e in db[pid].events if e.table == "condition"} & top30_set
            for a, b in itertools.combinations(sorted(codes_here), 2):
                joint_counts[(a, b)] = joint_counts.get((a, b), 0) + 1

        edge_rows = []
        for (a, b), joint in joint_counts.items():
            if joint < 5:  # drop pairs too rare to trust a ratio from
                continue
            p_a, p_b, p_ab = overall[a] / n_cohort, overall[b] / n_cohort, joint / n_cohort
            edge_rows.append((a, b, joint, p_ab / (p_a * p_b)))
        edges_df = pd.DataFrame(edge_rows, columns=["a", "b", "joint", "lift"])
        top_edges = edges_df.sort_values("lift", ascending=False).head(60)

        G = nx.Graph()
        for c in top30_codes:
            G.add_node(c, name=resolved_all["concept_name"].get(c, c), weight=int(overall[c]))
        for _, r in top_edges.iterrows():
            G.add_edge(r["a"], r["b"], weight=r["lift"])
        G.remove_nodes_from(list(nx.isolates(G)))

        fig, ax = plt.subplots(figsize=(11, 11))
        layout = nx.kamada_kawai_layout(G)
        sizes = [G.nodes[n]["weight"] for n in G.nodes]
        nx.draw_networkx_nodes(G, layout, node_size=sizes, node_color="#4C72B0", alpha=0.85, ax=ax)
        widths = [G[u][v]["weight"] * 0.8 for u, v in G.edges]
        nx.draw_networkx_edges(G, layout, width=widths, alpha=0.4, edge_color="#888888", ax=ax)
        nx.draw_networkx_labels(
            G,
            layout,
            labels={n: G.nodes[n]["name"][:26] for n in G.nodes},
            font_size=7,
            ax=ax,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=0.5),
        )
        ax.set_title(
            f"{task}: top-30 diagnosis co-occurrence preview, lift-weighted top 60 edges\n"
            "(node size = patient count, edge width = lift, isolated nodes dropped)"
        )
        ax.axis("off")
        fig.tight_layout()
        plt.show()
        print(
            f"co-occurrence network: {G.number_of_nodes()} nodes (of 30), {G.number_of_edges()} "
            "edges -- this is a preview, not the disease-trajectory graph itself (that's issue "
            "#10: directed, relative-risk-weighted, top ~200 concepts, symmetrized for "
            "Girvan-Newman)."
        )

    return demo, top20_df

### `guo_los` (Long Length of Stay) -- our primary task

In [ ]:
demo_los, top20_los = zoom_in("guo_los", build_cooccurrence_network=True)

### `new_hypertension` -- our secondary task

In [ ]:
demo_htn, top20_htn = zoom_in("new_hypertension", build_cooccurrence_network=False)

## Summary -- issue #3 done-when checklist

- **Both tasks' label/split counts confirmed against our own copy of the data, not just PROJECT_GUIDE**: `guo_los` 3,855 patients / 1,271 ever-positive, `new_hypertension` 2,328 patients / 386 ever-positive -- exact matches, asserted above, not eyeballed.
- **Label rows vs. patient counts resolved**: for both locked tasks, `n_rows > n_patients` -- a patient can have several `prediction_time` rows with different label values (e.g. multiple hospital stays, each independently long/short). This confirms `(patient_id, prediction_time)` is the real node identity for graph construction, not `patient_id` alone -- directly informs how issue #4's `build_patient_vectors` needs to key its output.
- **Official train/val/test splits present and complete** for every patient in both tasks (no missing split assignment).
- **Diagnosis-code name resolution via a real OMOP join** (`concept.csv`, not a heuristic pip package): 100% match rate across all distinct condition codes in both cohorts' full histories.
- **Genuine, reportable signal, not just plumbing**: ever-positive patients are older on average in both tasks (mirrors known clinical risk factors for long stays / new hypertension), and the top-diagnosis comparison shows a plausible cardiometabolic comorbidity skew (hypertension, hyperlipidemia, abnormal ECG) in the `guo_los`-positive cohort specifically.
- **Reminder for every later notebook**: everything in Part 3 used *full patient history*, not prediction_time-censored history. The leakage-safe condition (issue #6) is a separate, deliberately different code path (`events_before`, not `events_all`).